# Part 1 — chi² ranking via Spark RDDs

This notebook reproduces Assignment 1's per-category top-K chi² ranking using the **Spark RDD API**, producing `outputs/output_rdd.txt` with line format identical to Assignment 1's `output.txt`.

All pipeline logic lives in [chi_square_rdd.py](chi_square_rdd.py); this notebook is a thin driver that imports and calls it, so the script and notebook stay in lock-step.

## Pipeline overview

1. Read JSON-lines reviews from local FS or HDFS.
2. Tokenize `reviewText` (regex split, casefold, drop short words and stopwords, **deduplicate per document** — chi² uses presence, not frequency).
3. Cache the `(category, deduped_tokens)` RDD; reuse for `N`, `n_c`, and the tagged-counts emission.
4. One `flatMap` emits both `("T", term)` and `("TC", term, cat)` keys → one `reduceByKey` produces both `n_t` and `n_tc` count families.
5. `join` on `term` to attach `n_t` to each `(term, cat)` row, then map to chi² scores.
6. `aggregateByKey` with a bounded min-heap (size 75) selects top-K per category — single shuffle.
7. Driver-side formatter writes the 23-line output (22 category lines + alphabetical merged dictionary).

## Imports

We make `src/` importable so both `common/` and `part1_rdd/` resolve.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
SRC_DIR = (NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'part1_rdd' else NOTEBOOK_DIR / 'src')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from part1_rdd import chi_square_rdd

## Local run against the dev sample

Adjust paths if your checkout layout differs. The dev sample lives in Assignment 1's assets folder.

In [ ]:
ASSIGNMENT_2 = SRC_DIR.parent
INPUT_PATH = str(ASSIGNMENT_2.parent / 'Assignment 1' / 'src' / 'Assignment_1_Assets' / 'reviews_devset.json')
STOPWORDS_PATH = str(ASSIGNMENT_2 / 'src' / 'common' / 'stopwords.txt')
OUTPUT_PATH = str(ASSIGNMENT_2 / 'outputs' / 'output_rdd.txt')

print('input    :', INPUT_PATH)
print('stopwords:', STOPWORDS_PATH)
print('output   :', OUTPUT_PATH)

In [ ]:
chi_square_rdd.run(
    input_path=INPUT_PATH,
    stopwords_path=STOPWORDS_PATH,
    output_path=OUTPUT_PATH,
    mode='local',
    top_k=75,
)

## Quick sanity checks

- Total line count should be 23 (22 categories + 1 merged-dictionary line).
- Each category line should have exactly 75 `term:score` tokens after the category name.

In [ ]:
lines = Path(OUTPUT_PATH).read_text(encoding='utf-8').splitlines()
print(f'lines: {len(lines)}')
for line in lines[:-1]:
    cat, *terms = line.split(' ')
    print(f'  {cat:30s} {len(terms)} terms')
print(f'merged dict: {len(lines[-1].split())} unique terms')

## Comparison vs Assignment 1

Per-line term-set overlap with `../Assignment 1/output.txt`. Score values may differ in trailing digits due to non-deterministic reduce order across frameworks; term sets should match almost completely.

In [ ]:
REF_PATH = ASSIGNMENT_2.parent / 'Assignment 1' / 'output.txt'
ours = Path(OUTPUT_PATH).read_text(encoding='utf-8').splitlines()
ref  = REF_PATH.read_text(encoding='utf-8').splitlines()

def parse_terms(line):
    cat, *pairs = line.split(' ')
    terms = {p.rsplit(':', 1)[0] for p in pairs}
    return cat, terms

for ours_line, ref_line in zip(ours[:-1], ref[:-1]):
    cat_a, terms_a = parse_terms(ours_line)
    cat_b, terms_b = parse_terms(ref_line)
    assert cat_a == cat_b, (cat_a, cat_b)
    overlap = len(terms_a & terms_b)
    print(f'  {cat_a:30s} overlap {overlap:3d}/75   missing-from-ours {sorted(terms_b - terms_a)[:3]}')

## Cluster run

Don't run on the cluster from inside this notebook — use `spark-submit` from a terminal so the job goes to YARN cluster mode and reads from HDFS:

```bash
( cd src && zip -r ../common.zip common )
spark-submit --master yarn --deploy-mode cluster \
  --py-files common.zip \
  --files src/common/stopwords.txt \
  src/part1_rdd/chi_square_rdd.py \
  --mode cluster \
  --input hdfs:///dic_shared/amazon-reviews/full/reviews_devset.json \
  --stopwords stopwords.txt \
  --output output_rdd.txt
```